# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6)

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Model
from keras.layers import Dense, Input, LSTM, Flatten, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


2026-03-05 12:01:25.150203: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
#N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
N_INPUT_LIST = [42, 48]
HORIZONS = [5]

# Výstupné priečinky
MODELS_DIR = "models_3atr"
RESULTS_DIR = "results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
train_raw

,Unnamed: 0.1,Unnamed: 0,time1,bz_gsm,v,DST,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
0,0,0,1963-01-01 00:30:00+00:00,-0.2,285.0,-6,-6,-5.0,-5.0,-3.0,-3.0,-6.0
1,1,1,1963-01-01 01:30:00+00:00,-0.2,285.0,-5,-5,-5.0,-3.0,-3.0,-6.0,-8.0
2,2,2,1963-01-01 02:30:00+00:00,-0.2,285.0,-5,-5,-3.0,-3.0,-6.0,-8.0,-9.0
3,3,3,1963-01-01 03:30:00+00:00,-0.2,285.0,-3,-3,-3.0,-6.0,-8.0,-9.0,-6.0
4,4,4,1963-01-01 04:30:00+00:00,-0.2,285.0,-3,-3,-6.0,-8.0,-9.0,-6.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
286690,286690,286690,1995-09-15 10:30:00+00:00,-5.6,429.0,-52,-52,-48.0,-40.0,-39.0,-45.0,-48.0
286691,286691,286691,1995-09-15 11:30:00+00:00,-2.9,435.0,-48,-48,-40.0,-39.0,-45.0,-48.0,-45.0
286692,286692,286692,1995-09-15 12:30:00+00:00,-7.8,440.0,-40,-40,-39.0,-45.0,-48.0,-45.0,-41.0
286693,286693,286693,1995-09-15 13:30:00+00:00,-5.6,440.0,-39,-39,-45.0,-48.0,-45.0,-41.0,-41.0


In [6]:
def build_model(n_input: int) -> keras.Model:
    """Architektúra z pôvodného notebooku."""
    inputs = Input(shape=(n_input, 3))
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(inputs)
    x = LSTM(128, return_sequences=True)(x)
    x = TimeDistributed(Dense(1, activation="linear"))(x)
    x = Flatten()(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer="adam", metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    x_cols = ["DST", "bz_gsm", "v"]
    features = x_cols + [y_col]

    train = train_df[features].copy()
    test  = test_df[features].copy()

    # odstránenie NaN spôsobených DST+X
    train = train.dropna().reset_index(drop=True)
    test  = test.dropna().reset_index(drop=True)

    # časový split train/valid
    valid_size = int(len(train) * 0.2)
    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    # X musí byť 2D: (počet_riadkov, 3)
    X_train = train[x_cols].values
    y_train = train[y_col].values

    X_val = valid[x_cols].values
    y_val = valid[y_col].values

    X_test = test[x_cols].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)



In [7]:
def train_one(y_col: str, n_input: int):
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits(train_raw, test_raw, y_col)

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    
    model = build_model(n_input)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H_BZ+V.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    # finálne vyhodnotenie na test sete
    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    # uložiť aj history (voliteľné)
    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H_BZ+V.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }


In [ ]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary_5_BZ+V.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+5, n_input=42
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 880ms/step - loss: 446.9039 - mae: 14.2412
Epoch 1: val_mae improved from inf to 12.30931, saving model to models_3atr/DST+5_42H_BZ+V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 872s 956ms/step - loss: 446.8011 - mae: 14.2390 - val_loss: 445.5724 - val_mae: 12.3093
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - loss: 232.8388 - mae: 9.6626
Epoch 2: val_mae improved from 12.30931 to 10.15450, saving model to models_3atr/DST+5_42H_BZ+V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 859s 958ms/step - loss: 232.8078 - mae: 9.6620 - val_loss: 309.9921 - val_mae: 10.1545
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - loss: 195.2501 - mae: 8.9064
Epoch 3: val_mae improved from 10.15450 to 9.89899, saving model to models_3atr/DST+5_42H_BZ+V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 872s 973ms/step - loss: 195.2397 - mae: 8.9061 - val_loss: 261.5189 - val_mae: 9.8990
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 874ms/step - loss: 177.7691 - mae: 8.4194
Epoch 4: val_mae im

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


 41/896 ━━━━━━━━━━━━━━━━━━━━ 13:44 965ms/step - loss: 651.2993 - mae: 16.1792

## Poznámky
- Ak chceš presne poradie ako si písala (najprv `DST+1` pre všetky `n_input`, potom `DST+2`, …), tak to presne robí horný loop (horizonty vonkajší, `n_input` vnútorný).
- Ak chceš opačne (pre dané `n_input` spraviť `DST+1..6`), stačí prehodiť poradie cyklov.
